### Collect data using api endpoints and store them in a database

#### Endpoint - https://clinicaltrials.gov/api/v2

#### BASIC Fetching

In [1]:
import httpx
import asyncio

In [2]:
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

* async with block ensures the client's connection resources are properly opened and cleaned up (closed) automatically when the block exits, even if an error occurs.

In [3]:
async def fetch_trials():
    url = BASE_URL

    async with httpx.AsyncClient() as client: # Creates an async HTTP client using the httpx library (a modern alternative to requests)
        response = await client.get(url) #  Sends an asynchronous GET request to the specified URL and waits for the response.
        response.raise_for_status() # Checks the HTTP status code. If it's an error this raises an exception and If the request succeeded, it does nothing and execution continues.
        data = response.json() # Parses the JSON response into a Python dictionary.
        return data
    

In [ ]:
async def main():
    data = await fetch_trials()

    print(data)

asyncio.run(main())

### FETCHING + PARAMS

* response = await client.get(url, params=params)

In [ ]:
async def fetch_trials():
    url = BASE_URL

    params = {
        "query.cond": "diabetes",
        "pageSize": 10
    }

    async with httpx.AsyncClient() as client: # Creates an async HTTP client using the httpx library (a modern alternative to requests)
        response = await client.get(url, params=params) #  Sends an asynchronous GET request to the specified URL and waits for the response.

        response.raise_for_status() # Checks the HTTP status code. If it's an error this raises an exception and If the request succeeded, it does nothing and execution continues.
        data = response.json() # Parses the JSON response into a Python dictionary.
        return data

In [ ]:
async def main():
    data = await fetch_trials()

    print(data)


asyncio.run(main())

### Fetching data Using Fast api

In [ ]:
from fastapi import FastAPI
import httpx

In [ ]:
app = FastAPI()

In [ ]:
@app.get("/trials")
async def get_trials():

    url = "https://clinicaltrials.gov/api/v2/studies"

    params = {
        "query.cond": "diabetes",
        "pageSize": 10
    }

    async with httpx.AsyncClient() as client:

        response = await client.get(
            url,
            params=params
        )

        response.raise_for_status()

        data = response.json()

    return data

* Seperate and fetching logic and fast api == cleaner`

In [ ]:
app = FastAPI()


async def fetch_trials(condition: str, page_size: int = 10):

    url = "https://clinicaltrials.gov/api/v2/studies"

    params = {
        "query.cond": condition,
        "pageSize": page_size
    }

    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)

        response.raise_for_status()

        return response.json()


@app.get("/trials")
async def get_trials(condition: str, page_size: int = 10):   # dynamic condition (diabetes, cancer, etc.) and page size (number of trials to fetch)

    data = await fetch_trials(condition, page_size)

    return data

* Extract a Specific Field = studies

In [ ]:
@app.get("/trials")
async def get_trials(condition: str):

    data = await fetch_trials(condition)

    return data["studies"]

* Extract Multiple sub-Fields

In [ ]:
@app.get("/trials")
async def get_trials(condition: str):

    data = await fetch_trials(condition)

    results = []

    for study in data["studies"]:

        protocol = study["protocolSection"]

        identification = protocol["identificationModule"]
        status = protocol["statusModule"]

        results.append({
            "nct_id": identification.get("nctId"),
            "title": identification.get("briefTitle"),
            "status": status.get("overallStatus")
        })

    return results

### Runtime Exceptions

In [ ]:
from fastapi import HTTPException

In [ ]:
@app.get("/trials")
async def get_trials():

    url = "https://clinicaltrials.gov/api/v2/studies"

    params = {
        "query.cond": "diabetes",
        "pageSize": 10
    }

    try:
        async with httpx.AsyncClient(timeout=10.0) as client:

            response = await client.get(
                url,
                params=params
            )

            response.raise_for_status()

            data = response.json()

    except httpx.TimeoutException:
        raise HTTPException(status_code=504, detail="Request timed out")

    except httpx.ConnectError:
        raise HTTPException(status_code=503, detail="Could not connect to upstream service")

    except httpx.HTTPStatusError as e:
        raise HTTPException(status_code=e.response.status_code, detail="Upstream returned an error")

    except httpx.RequestError:
        raise HTTPException(status_code=502, detail="Error contacting upstream service")

    except ValueError:
        raise HTTPException(status_code=502, detail="Upstream returned invalid JSON")

    except Exception:
        raise HTTPException(status_code=500, detail="Internal server error")

    return data